In [5]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from typing import TypedDict,Annotated,Literal
from pydantic import BaseModel,Field

In [6]:
load_dotenv()

llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",
    temperature=0
)

In [7]:
class SentimentSchema(BaseModel):
    sentiment: Literal['positive','negative']=Field(description="Sentiment of the review")
    

In [14]:
class DiagnosisSchema(BaseModel):
    issue_type: Literal["UX", "Performance", "Bug", "Support", "Other"] = Field(description='The category of issue mentioned in the review')
    tone: Literal["angry", "frustrated", "disappointed", "calm"] = Field(description='The emotional tone expressed by the user')
    urgency: Literal["low", "medium", "high"] = Field(description='How urgent or critical the issue appears to be')

In [15]:
structured_model=llm.with_structured_output(SentimentSchema)
structured_model2=llm.with_structured_output(DiagnosisSchema)

In [16]:
class ReviewState(TypedDict):
    review: str
    sentiment: Literal['positive','negative']
    diagnosis: dict
    response: str

In [37]:
def find_sentiment(state:ReviewState):
    prompt=f"For the following review find the sentiment \n {state['review']}"
    sentiment=structured_model.invoke(prompt)
    return {'sentiment':sentiment}

def positive_response(state:ReviewState):
    prompt = f"""Write a warm thank-you message in response to this review:
    \n\n\"{state['review']}\"\nAlso, kindly ask the user to leave feedback on our website."""
    response=llm.invoke(prompt).content
    return {'response':response}

def run_diagnosis(state:ReviewState):
    prompt = f"""Diagnose this negative review:\n\n{state['review']}\n
    Return issue_type, tone, and urgency."""
    response=structured_model2.invoke(prompt)
    return {'diagnosis':response.model_dump()}

def negative_response(state:ReviewState):
    diagnosis=state['diagnosis']
    prompt = f"""You are a support assistant.
The user had a '{diagnosis['issue_type']}' issue, sounded '{diagnosis['tone']}', and marked urgency as '{diagnosis['urgency']}'.
Write an empathetic, helpful resolution message."""
    response=llm.invoke(prompt).content
    return {'response':response}

def check_sentiment(state:ReviewState) -> Literal['positive_response','run_diagnosis']:
    if state['sentiment']=='positive':
        return 'positive_response'
    else:
        return 'run_diagnosis'

In [38]:
graph=StateGraph(ReviewState)

graph.add_node('find_sentiment',find_sentiment)
graph.add_node('positive_response',positive_response)
graph.add_node('run_diagnosis',run_diagnosis)
graph.add_node('negative_response',negative_response)

graph.add_edge(START,'find_sentiment')
graph.add_conditional_edges('find_sentiment',check_sentiment)
graph.add_edge('positive_response',END)
graph.add_edge('run_diagnosis','negative_response')
graph.add_edge('negative_response',END)

workflow=graph.compile()

In [39]:
intial_state={
    'review': "I've been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)

{'review': "I've been trying to log in for over an hour now, and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality.",
 'sentiment': SentimentSchema(sentiment='negative'),
 'diagnosis': {'issue_type': 'Bug', 'tone': 'frustrated', 'urgency': 'high'},
 'response': "Subject: Re: High Urgency Bug Report - We're on it!\n\nHi [User Name],\n\nI'm so sorry to hear you're encountering this bug, and I completely understand how incredibly frustrating it must be, especially given the high urgency you've indicated. It sounds like it's really disrupting your workflow, and for that, I sincerely apologize.\n\nPlease be assured that we're treating this with the highest priority. I've immediately logged this issue with our engineering team and escalated it for urgent investigation. They are already looking into the details you provided.\n\nTo help us diagnose and resolve this as 